# 0. Import Libraries

In [ ]:
import os, sys
from pathlib import Path

def add_project_path(module_folder="model_implementation"):
    candidates = [
        os.path.abspath("."),
        os.path.abspath("../src"),
        os.path.abspath("src"),
    ]

    for path in candidates:
        if os.path.exists(os.path.join(path, module_folder)):
            if path not in sys.path:
                sys.path.append(path)
            return path

    raise ImportError(f"Could not find '{module_folder}' in current or parent directory")

SRC_PATH = add_project_path("model_implementation")
add_project_path("cnn")
add_project_path("rnn")
ROOT = Path(SRC_PATH).parent.resolve()
print("ROOT:", ROOT)

In [ ]:
# Jalankan cell ini hanya bila environment belum memiliki dependency.
# import subprocess
# subprocess.check_call([sys.executable, "-m", "pip", "install", "tensorflow", "scikit-learn", "pandas", "matplotlib", "nltk", "pillow"])

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import tensorflow as tf
except Exception as exc:
    tf = None
    print("TensorFlow belum tersedia:", exc)

from cnn.utility import feature_extractor, image_paths
from common.io import save_json, load_json, load_npy
from rnn.preprocess import prep_data, load_vocab
from rnn.sequences import align_features_to_captions
from rnn.decode import decode_batch
from rnn.caption_decoder import build_decoder
from rnn.evaluate import eval_keras as eval_caption_keras, eval_caption_decoder, seq_tokens

# 1. Global Variables

In [ ]:
SEED = 42
IMAGE_SIZE = (150, 150)
BATCH_SIZE = 32
EPOCHS = 10
MAX_CAPTION_LENGTH = 34

np.random.seed(SEED)
plt.style.use("seaborn-v0_8")

DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
FEATURE_DIR = DATA_DIR / "features"
VOCAB_DIR = DATA_DIR / "vocab"
MODEL_DIR = ROOT / "models"
CNN_MODEL_DIR = MODEL_DIR / "cnn"
RNN_MODEL_DIR = MODEL_DIR / "rnn"
REPORT_DIR = ROOT / "reports"
TABLE_DIR = REPORT_DIR / "tables"
FIG_DIR = REPORT_DIR / "figures"

for path in [FEATURE_DIR, VOCAB_DIR, CNN_MODEL_DIR, RNN_MODEL_DIR, TABLE_DIR, FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CATEGORIES = {
    "buildings": 0,
    "forest": 1,
    "glacier": 2,
    "mountain": 3,
    "sea": 4,
    "street": 5,
}
INV_CATEGORIES = {v: k for k, v in CATEGORIES.items()}

if tf is not None:
    gpu_devices = tf.config.list_physical_devices("GPU")
    if gpu_devices:
        print("GPU Detected:", gpu_devices)
        for gpu in gpu_devices:
            tf.config.experimental.set_memory_growth(gpu, True)
    else:
        print("No GPU found, defaulting to CPU.")

# 2. Caption Artifacts

In [ ]:
def read_flickr8k_caption_pairs(path):
    pairs = []
    with Path(path).open('r', encoding='utf-8') as file:
        for line_number, line in enumerate(file):
            line = line.strip()
            if not line:
                continue
            if line_number == 0 and line.lower().startswith('image,'):
                continue
            if ',' in line:
                image_name, caption = line.split(',', 1)
            else:
                image_name, caption = line.split(None, 1)
            image_id = Path(image_name.split('#')[0]).stem
            pairs.append((image_id, caption))
    return pairs

CAPTION_FILE = RAW_DIR / "flickr8k" / "captions.txt"
CAPTION_PATH = VOCAB_DIR / "caption_sequences.npy"
VOCAB_PATH = VOCAB_DIR / "vocab.json"
CAPTION_ID_PATH = VOCAB_DIR / "caption_image_ids.txt"

caption_sequences = None
caption_image_ids = []
word_to_index = index_to_word = None

if CAPTION_FILE.exists():
    caption_pairs = read_flickr8k_caption_pairs(CAPTION_FILE)
    caption_image_ids = [image_id for image_id, _ in caption_pairs]
    captions = [caption for _, caption in caption_pairs]
    caption_sequences, word_to_index, index_to_word = prep_data(captions, max_length=MAX_CAPTION_LENGTH, out_dir=VOCAB_DIR)
    CAPTION_ID_PATH.write_text('\n'.join(caption_image_ids), encoding='utf-8')
    print("caption pairs:", len(caption_pairs))
    print("unique images:", len(set(caption_image_ids)))
    print("sequences:", caption_sequences.shape)
    print("vocab:", len(word_to_index))
elif CAPTION_PATH.exists() and VOCAB_PATH.exists() and CAPTION_ID_PATH.exists():
    caption_sequences = load_npy(CAPTION_PATH).astype('int32')
    caption_image_ids = [line.strip() for line in CAPTION_ID_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
    word_to_index, index_to_word = load_vocab(VOCAB_DIR)
    print("Loaded caption artifacts:", caption_sequences.shape, len(caption_image_ids), len(word_to_index))
else:
    print("Caption artifacts belum tersedia.")

# 3. Feature Artifacts

In [ ]:
IMAGE_DIR = RAW_DIR / "flickr8k" / "Images"
FEATURE_PATH = FEATURE_DIR / "flickr8k_features.npy"
FEATURE_ID_PATH = FEATURE_DIR / "flickr8k_image_ids.txt"
ENCODER_PATH = CNN_MODEL_DIR / "flickr8k_inceptionv3_encoder.keras"

features = None
feature_image_ids = []

if tf is not None and IMAGE_DIR.exists():
    img_paths = image_paths(IMAGE_DIR)
    feature_image_ids = [path.stem for path in img_paths]
    if ENCODER_PATH.exists():
        encoder = tf.keras.models.load_model(ENCODER_PATH)
    else:
        encoder = tf.keras.applications.InceptionV3(include_top=False, weights='imagenet', input_shape=(299, 299, 3), pooling='avg')
        encoder.trainable = False
        encoder.save(ENCODER_PATH)
    features = feature_extractor(img_paths, encoder, FEATURE_PATH, target_size=encoder.input_shape[1:3], batch_size=BATCH_SIZE, image_id_path=FEATURE_ID_PATH, preprocess_fn=tf.keras.applications.inception_v3.preprocess_input)
    print("images:", len(feature_image_ids))
    print("features:", features.shape)
elif FEATURE_PATH.exists() and FEATURE_ID_PATH.exists():
    features = load_npy(FEATURE_PATH).astype('float32')
    feature_image_ids = [line.strip() for line in FEATURE_ID_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
    print("Loaded existing features:", features.shape, len(feature_image_ids))
else:
    print("Feature extraction dilewati.")

# 4. RNN/LSTM Evaluation

In [ ]:
keras_scores = []
manual_scores = []

if not rnn_records and (TABLE_DIR / 'train_records.json').exists():
    rnn_records = load_json(TABLE_DIR / 'train_records.json')
if features is None and FEATURE_PATH.exists() and FEATURE_ID_PATH.exists():
    features = load_npy(FEATURE_PATH).astype('float32')
    feature_image_ids = [line.strip() for line in FEATURE_ID_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
if caption_sequences is None and CAPTION_PATH.exists() and VOCAB_PATH.exists() and CAPTION_ID_PATH.exists():
    caption_sequences = load_npy(CAPTION_PATH).astype('int32')
    caption_image_ids = [line.strip() for line in CAPTION_ID_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
    word_to_index, index_to_word = load_vocab(VOCAB_DIR)
if 'aligned_features' not in globals() or aligned_features is None:
    if features is not None and caption_sequences is not None and caption_image_ids and feature_image_ids:
        aligned_features, aligned_captions, aligned_caption_ids, missing_caption_ids = align_features_to_captions(features, feature_image_ids, caption_sequences, caption_image_ids)

if tf is not None and rnn_records and aligned_features is not None and aligned_captions is not None:
    max_len = int(aligned_captions.shape[1] - 1)
    for record in rnn_records:
        cfg = record['config']
        model = tf.keras.models.load_model(record['model_path'])
        score = eval_caption_keras(model, aligned_features, aligned_captions, word_to_index, index_to_word, max_len)
        score.update({'implementation': 'keras', 'recur_type': cfg['recur_type'], 'recur_layers': cfg['recur_layers'], 'hidden_size': cfg['hidden_size'], 'model_path': record['model_path']})
        keras_scores.append(score)

        decoder = build_decoder(cfg, record['weight_path'])
        manual_score = eval_caption_decoder(decoder, aligned_features, aligned_captions, word_to_index, index_to_word, max_len)
        manual_score.update({'implementation': 'scratch_numpy', 'recur_type': cfg['recur_type'], 'recur_layers': cfg['recur_layers'], 'hidden_size': cfg['hidden_size'], 'weight_path': record['weight_path']})
        manual_scores.append(manual_score)

    save_json(keras_scores, TABLE_DIR / 'rnn_scores.json')
    save_json(manual_scores, TABLE_DIR / 'manual_scores.json')
    # Tabel turunan untuk kebutuhan analisis laporan.
    keras_df = pd.DataFrame(keras_scores)
    manual_df = pd.DataFrame(manual_scores)
    if not keras_df.empty:
        keras_df.to_csv(TABLE_DIR / 'rnn_scores.csv', index=False)
        keras_df.groupby(['recur_type', 'recur_layers'], as_index=False)[['bleu4', 'meteor', 'runtime_seconds']].mean().to_csv(TABLE_DIR / 'caption_by_recurrent_layers.csv', index=False)
        keras_df.groupby(['recur_type', 'hidden_size'], as_index=False)[['bleu4', 'meteor', 'runtime_seconds']].mean().to_csv(TABLE_DIR / 'caption_by_hidden_size.csv', index=False)
        keras_df.groupby('recur_type', as_index=False)[['bleu4', 'meteor', 'runtime_seconds']].mean().to_csv(TABLE_DIR / 'rnn_vs_lstm.csv', index=False)
        keras_df[keras_df['recur_type'] == 'rnn'].groupby('recur_layers', as_index=False)[['bleu4', 'meteor', 'runtime_seconds']].mean().to_csv(TABLE_DIR / 'rnn_layer_comparison.csv', index=False)
        keras_df[keras_df['recur_type'] == 'lstm'].groupby('recur_layers', as_index=False)[['bleu4', 'meteor', 'runtime_seconds']].mean().to_csv(TABLE_DIR / 'lstm_layer_comparison.csv', index=False)
        keras_df[keras_df['recur_type'] == 'rnn'].groupby('hidden_size', as_index=False)[['bleu4', 'meteor', 'runtime_seconds']].mean().to_csv(TABLE_DIR / 'rnn_hidden_comparison.csv', index=False)
        keras_df[keras_df['recur_type'] == 'lstm'].groupby('hidden_size', as_index=False)[['bleu4', 'meteor', 'runtime_seconds']].mean().to_csv(TABLE_DIR / 'lstm_hidden_comparison.csv', index=False)
    if not manual_df.empty:
        manual_df.to_csv(TABLE_DIR / 'manual_scores.csv', index=False)
    if not keras_df.empty and not manual_df.empty:
        impl_df = pd.concat([keras_df, manual_df], ignore_index=True)
        impl_df.groupby(['implementation', 'recur_type'], as_index=False)[['bleu4', 'meteor', 'runtime_seconds']].mean().to_csv(TABLE_DIR / 'keras_vs_scratch.csv', index=False)
    if keras_scores:
        score_df = pd.DataFrame(keras_scores)
        labels = score_df['recur_type'].astype(str) + '-' + score_df['recur_layers'].astype(str) + 'x-' + score_df['hidden_size'].astype(str)
        plt.figure(figsize=(12, 4))
        plt.bar(labels, score_df['bleu4'])
        plt.xticks(rotation=45, ha='right')
        plt.ylabel('BLEU-4')
        plt.tight_layout()
        plt.savefig(FIG_DIR / 'rnn_bleu4.png', dpi=160)
        plt.show()
    for record in rnn_records:
        history_path = Path(record.get('history_path', ''))
        if not history_path.exists():
            continue
        history = load_json(history_path)
        name = Path(record.get('model_path', 'model')).stem
        plt.figure(figsize=(6, 4))
        if 'loss' in history:
            plt.plot(history['loss'], label='train loss')
        if 'val_loss' in history:
            plt.plot(history['val_loss'], label='validation loss')
        plt.title(name)
        plt.xlabel('epoch')
        plt.ylabel('loss')
        plt.legend()
        plt.tight_layout()
        plt.savefig(FIG_DIR / f'{name}_loss.png', dpi=160)
        plt.close()
    if keras_scores:
        score_df = pd.DataFrame(keras_scores)
        labels = score_df['recur_type'].astype(str) + '-' + score_df['recur_layers'].astype(str) + 'x-' + score_df['hidden_size'].astype(str)
        plt.figure(figsize=(12, 4))
        plt.bar(labels, score_df['bleu4'])
        plt.xticks(rotation=45, ha='right')
        plt.ylabel('BLEU-4')
        plt.tight_layout()
        plt.savefig(FIG_DIR / 'rnn_bleu4.png', dpi=160)
        plt.show()
else:
    print("Evaluasi RNN/LSTM dilewati.")

pd.DataFrame(keras_scores)

# 5. Caption Length Study and Qualitative Analysis

In [ ]:
length_scores = []
qualitative_rows = []

if keras_scores and aligned_features is not None and aligned_captions is not None:
    best = max(keras_scores, key=lambda row: row.get('bleu4', 0.0))
    model = tf.keras.models.load_model(best['model_path'])
    for max_len in [10, 20, int(aligned_captions.shape[1] - 1)]:
        score = eval_caption_keras(model, aligned_features, aligned_captions, word_to_index, index_to_word, max_len)
        score.update({'max_length': max_len, 'model_path': best['model_path']})
        length_scores.append(score)
    save_json(length_scores, TABLE_DIR / 'length_scores.json')
    pd.DataFrame(length_scores).to_csv(TABLE_DIR / 'length_scores.csv', index=False)
    if length_scores:
        length_df = pd.DataFrame(length_scores)
        plt.figure(figsize=(6, 4))
        plt.plot(length_df['max_length'], length_df['bleu4'], marker='o')
        plt.xlabel('Max caption length')
        plt.ylabel('BLEU-4')
        plt.tight_layout()
        plt.savefig(FIG_DIR / 'length_bleu4.png', dpi=160)
        plt.show()

    best_by_kind = {}
    for score in keras_scores:
        kind = score.get('recur_type')
        if kind and (kind not in best_by_kind or score.get('bleu4', 0.0) > best_by_kind[kind].get('bleu4', 0.0)):
            best_by_kind[kind] = score
    models_by_kind = {kind: tf.keras.models.load_model(item['model_path']) for kind, item in best_by_kind.items()}
    # Pilih contoh kualitas tinggi/sedang/rendah berdasarkan BLEU rata-rata prediksi model terbaik per kind.
    candidate_count = min(100, len(aligned_features))
    candidate_indices = list(range(candidate_count))
    candidate_predictions = {
        kind: decode_batch(model, aligned_features[candidate_indices], word_to_index, index_to_word, int(aligned_captions.shape[1] - 1))
        for kind, model in models_by_kind.items()
    }
    scored_candidates = []
    for pos, idx in enumerate(candidate_indices):
        refs = [seq_tokens(aligned_captions[idx], index_to_word)]
        scores = [bleu4_score(refs, preds[pos]) for preds in candidate_predictions.values()]
        scored_candidates.append((idx, float(np.mean(scores)) if scores else 0.0))
    scored_candidates = sorted(scored_candidates, key=lambda item: item[1])
    if len(scored_candidates) >= 10:
        low = [idx for idx, _ in scored_candidates[:3]]
        mid_start = max((len(scored_candidates) // 2) - 2, 0)
        mid = [idx for idx, _ in scored_candidates[mid_start:mid_start + 4]]
        high = [idx for idx, _ in scored_candidates[-3:]]
        selected = (low + mid + high)[:10]
    else:
        selected = [idx for idx, _ in scored_candidates]
    predictions_by_kind = {
        kind: decode_batch(model, aligned_features[selected], word_to_index, index_to_word, int(aligned_captions.shape[1] - 1))
        for kind, model in models_by_kind.items()
    }
    for pos, idx in enumerate(selected):
        row = {
            'index': int(idx),
            'image_id': aligned_caption_ids[idx] if idx < len(aligned_caption_ids) else '',
            'ground_truth': ' '.join(seq_tokens(aligned_captions[idx], index_to_word)),
        }
        for kind, predictions in predictions_by_kind.items():
            row[f'{kind}_prediction'] = ' '.join(predictions[pos])
        qualitative_rows.append(row)
    save_json(qualitative_rows, TABLE_DIR / 'qualitative_examples.json')
    pd.DataFrame(qualitative_rows).to_csv(TABLE_DIR / 'qualitative_examples.csv', index=False)
else:
    print("Length study dan qualitative analysis dilewati.")

pd.DataFrame(qualitative_rows)